# DiffDock — Pose Distribution Analysis

Characterises the distribution of predicted poses for a single complex.
Change `RESULTS_DIR` to analyse any run.

In [ ]:
import sys
sys.path.insert(0, '..')  # project root when running from notebooks/

from utils.distribution_analysis import (
    load_poses,
    compute_rmsd_matrix,
    cluster_poses,
    saturation_analysis,
    plot_rmsd_heatmap,
    torsion_rose_plots,
    view_poses_colored,
    run_full_analysis,
)
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# ── Only cell you need to change per run ──────────────────────────────────
RESULTS_DIR = '../results/batch_job/6d08'
CLUSTER_CUTOFF = 2.0  # Angstroms
# ──────────────────────────────────────────────────────────────────────────

## 1. Load poses & compute pairwise RMSD

Uses `spyrmsd.rmsd.symmrmsd` — symmetry-corrected, so symmetric ligands are handled correctly.

In [ ]:
poses = load_poses(RESULTS_DIR)
print(f"{len(poses)} poses loaded")
for p in poses:
    print(f"  rank{p.rank:2d}  confidence={p.confidence:.2f}")

In [ ]:
rmsd_matrix = compute_rmsd_matrix(poses)
np.set_printoptions(precision=2, suppress=True)
print("RMSD matrix (Å):")
print(rmsd_matrix)

## 2. Clustering

Complete-linkage hierarchical clustering with a 2 Å RMSD cutoff (standard in docking).
Cluster population ≈ mode weight.

In [ ]:
cluster_labels, linkage_matrix = cluster_poses(rmsd_matrix, cutoff=CLUSTER_CUTOFF)
n_clusters = len(set(cluster_labels))
print(f"{n_clusters} cluster(s) at {CLUSTER_CUTOFF} Å cutoff")
for p, cl in zip(poses, cluster_labels):
    print(f"  rank{p.rank:2d}  conf={p.confidence:.2f}  cluster={cl}")

## 3. Saturation analysis

Mean pairwise RMSD of top-N poses vs. N. Plateau → most modes found. Still rising → more samples needed.

In [ ]:
fig = saturation_analysis(poses, rmsd_matrix)
plt.show()

## 4. RMSD heatmap

Rows/columns reordered by dendrogram leaf order. Red lines = cluster boundaries.
Block structure → well-separated modes. Diffuse → broad unimodal distribution.

In [ ]:
fig = plot_rmsd_heatmap(rmsd_matrix, labels=cluster_labels)
plt.show()

## 5. 3D viewer — coloured by confidence

Blue = high confidence, red = low confidence.

In [ ]:
viewer = view_poses_colored(
    poses,
    results_dir=RESULTS_DIR,
    color_by='confidence',
)
viewer.show()

## 6. 3D viewer — coloured by cluster

In [ ]:
viewer = view_poses_colored(
    poses,
    results_dir=RESULTS_DIR,
    color_by='cluster',
    cluster_labels=cluster_labels,
)
viewer.show()

## 7. Torsional analysis

Circular histogram per rotatable bond. CV = circular variance (0 = perfectly constrained, 1 = uniform).

In [ ]:
fig = torsion_rose_plots(poses)
plt.show()

## One-shot: `run_full_analysis`

Convenience wrapper — runs everything above in one call.

In [ ]:
results = run_full_analysis(RESULTS_DIR, cutoff=CLUSTER_CUTOFF)
results['saturation_fig'].show()
results['heatmap_fig'].show()
results['torsion_fig'].show()